# Use Case Feature Engineering — Data Saham IDX

Notebook ini membuat fitur-fitur baru dari tabel `stock_summary` di Supabase untuk keperluan analisis dan model ML.

## Alur Kerja

```
stock_summary (Supabase)
        │
        ▼
[Tier 1] Fitur Harian
[Tier 2] Fitur Lag-Based
[Tier 3] Cross-Sectional Ranking
        │
        ▼
stock_features (Supabase) + CSV backup
```

## Fitur yang Dihitung

### Tier 1 — Fitur Harian
- `daily_return_pct`: return harian berdasarkan previous close.
- `intraday_return_pct`: return dari open ke close.
- `high_low_range_pct`: range high-low relatif terhadap previous.
- `spread` & `spread_pct`: selisih bid-offer absolut dan relatif.
- `bid_offer_imbalance`: ketidakseimbangan volume bid vs offer.
- `foreign_net`, `foreign_buy_ratio`, `foreign_sell_ratio`: aliran asing.
- `value_per_frequency`, `avg_trade_size`: metrik likuiditas.
- `market_cap_proxy`: proxy kapitalisasi pasar.

### Tier 2 — Fitur Lag-Based
- `sma_5`, `sma_10`: simple moving average.
- `ema_5`, `ema_10`: exponential moving average.
- `volatility_5d`, `volatility_10d`: volatilitas rolling.

### Tier 3 — Cross-Sectional Ranking
- `rank_change_pct`: peringkat return harian.
- `rank_volume`: peringkat volume.
- `rank_value`: peringkat nilai transaksi.
- `rank_foreign_net`: peringkat net asing.

---

## Cell [1] — Setup & Import

In [ ]:
import os
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Import modul feature engineering lokal
from feature_engineering.config import DB_URL, DB_MODE, SOURCE_TABLE, TARGET_TABLE, ALL_FEATURES
from feature_engineering.feature_engineering import (
    load_raw_data,
    build_features,
    save_to_database,
    export_csv,
)

# Pastikan matplotlib plots tampil di notebook
%matplotlib inline
sns.set_style("whitegrid")

print(f"[CONFIG] Database: {DB_MODE}")
print(f"[CONFIG] Source table: {SOURCE_TABLE}")
print(f"[CONFIG] Target table: {TARGET_TABLE}")

---

## Cell [2] — Load Data Mentah dari Supabase

In [ ]:
# Load semua data yang tersedia
df_raw = load_raw_data()

print(f"\nData mentah: {len(df_raw):,} baris × {len(df_raw.columns)} kolom")
print(f"Rentang tanggal: {df_raw['date'].min()} s/d {df_raw['date'].max()}")
print(f"Jumlah saham unik: {df_raw['stock_code'].nunique()}")

df_raw.head()

---

## Cell [3] — Build Features

In [ ]:
df_features = build_features(df_raw)

print(f"\nFitur yang dihasilkan: {len(df_features.columns)} kolom")
print("Kolom fitur:", df_features.columns.tolist())

df_features.head(10)

---

## Cell [4] — Statistik Deskriptif Fitur

In [ ]:
# Statistik deskriptif untuk fitur numerik
df_features[ALL_FEATURES].describe().T

---

## Cell [5] — Missing Values Analysis

In [ ]:
missing = df_features[ALL_FEATURES].isna().sum().sort_values(ascending=False)
missing_pct = (missing / len(df_features) * 100).round(2)

missing_df = pd.DataFrame({
    "missing_count": missing,
    "missing_pct": missing_pct
})

print(missing_df)

# Visualisasi missing values
plt.figure(figsize=(10, 6))
missing_pct.plot(kind='barh')
plt.title("Persentase Missing Values per Fitur")
plt.xlabel("Missing (%)")
plt.tight_layout()
plt.show()

---

## Cell [6] — Distribusi Return Harian

In [ ]:
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
sns.histplot(df_features['daily_return_pct'].dropna(), bins=50, kde=True)
plt.title("Distribusi Daily Return (%).")
plt.xlabel("Daily Return (%)")

plt.subplot(1, 2, 2)
sns.boxplot(x=df_features['daily_return_pct'].dropna())
plt.title("Boxplot Daily Return (%).")

plt.tight_layout()
plt.show()

---

## Cell [7] — Top 10 Saham Berdasarkan Net Foreign Flow

In [ ]:
# Ambil data tanggal terakhir
latest_date = df_features['date'].max()
df_latest = df_features[df_features['date'] == latest_date]

top_foreign = df_latest.nlargest(10, 'foreign_net')[['stock_code', 'foreign_net', 'daily_return_pct']]

plt.figure(figsize=(10, 6))
sns.barplot(data=top_foreign, x='foreign_net', y='stock_code', palette='viridis')
plt.title(f"Top 10 Saham Net Foreign Buy — {latest_date}")
plt.xlabel("Foreign Net")
plt.tight_layout()
plt.show()

top_foreign

---

## Cell [8] — Korelasi Antar Fitur

In [ ]:
# Pilih subset fitur untuk heatmap korelasi
corr_cols = [
    'daily_return_pct', 'intraday_return_pct', 'high_low_range_pct',
    'spread_pct', 'bid_offer_imbalance', 'foreign_net',
    'volume', 'value', 'frequency',
    'sma_5', 'sma_10', 'volatility_5d'
]

# Gabungkan fitur dengan data mentah untuk korelasi
df_merged = df_features.merge(
    df_raw[['date', 'stock_code', 'volume', 'value', 'frequency']],
    on=['date', 'stock_code'],
    how='left'
)

corr_matrix = df_merged[corr_cols].corr()

plt.figure(figsize=(12, 10))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', center=0, fmt='.2f')
plt.title("Korelasi Antar Fitur")
plt.tight_layout()
plt.show()

---

## Cell [9] — Contoh Time Series untuk 1 Saham

In [ ]:
# Pilih saham BBCA sebagai contoh (ganti kode saham jika tidak tersedia)
sample_code = 'BBCA'
df_sample = df_features[df_features['stock_code'] == sample_code].sort_values('date')

if not df_sample.empty:
    plt.figure(figsize=(14, 8))

    plt.subplot(2, 1, 1)
    plt.plot(df_sample['date'], df_sample['close'] if 'close' in df_sample else df_sample['sma_5'], label='Close (proxy)')
    plt.plot(df_sample['date'], df_sample['sma_5'], label='SMA 5', linestyle='--')
    plt.plot(df_sample['date'], df_sample['sma_10'], label='SMA 10', linestyle='--')
    plt.title(f"Harga & Moving Averages — {sample_code}")
    plt.legend()
    plt.xticks(rotation=45)

    plt.subplot(2, 1, 2)
    plt.bar(df_sample['date'], df_sample['foreign_net'], color='green' if (df_sample['foreign_net'] >= 0).all() else None)
    plt.title(f"Foreign Net Flow — {sample_code}")
    plt.ylabel("Foreign Net")
    plt.xticks(rotation=45)

    plt.tight_layout()
    plt.show()
else:
    print(f"Saham {sample_code} tidak ditemukan di dataset.")

---

## Cell [10] — Save ke Supabase & Export CSV

In [ ]:
# Simpan ke Supabase
save_to_database(df_features)

# Export CSV backup
csv_path = export_csv(df_features, suffix="notebook")

print(f"\n[SUCCESS] Fitur tersimpan di Supabase dan CSV: {csv_path}")

---

## Cell [11] — Verifikasi Data di Supabase

In [ ]:
from sqlalchemy import create_engine, text

engine = create_engine(DB_URL, echo=False)

with engine.connect() as conn:
    result = conn.execute(text(f"SELECT date, COUNT(*) as records FROM {TARGET_TABLE} GROUP BY date ORDER BY date"))
    df_verify = pd.DataFrame(result.fetchall(), columns=result.keys())

print("Jumlah record per tanggal di stock_features:")
print(df_verify)

with engine.connect() as conn:
    total = conn.execute(text(f"SELECT COUNT(*) FROM {TARGET_TABLE}")).scalar()
print(f"\nTotal record di stock_features: {total:,}")

---

## Ringkasan

Notebook ini telah:
1. Membaca data saham dari tabel `stock_summary` di Supabase.
2. Menghitung 22 fitur baru (Tier 1, Tier 2, Tier 3).
3. Melakukan eksplorasi dan visualisasi sederhana.
4. Menyimpan hasil ke tabel `stock_features` di Supabase.
5. Mengekspor hasil ke CSV sebagai backup.

Untuk production, gunakan script `feature_engineering.py` yang bisa dijalankan via command line atau GitHub Actions.